# Myntra: Data-driven fashion returns reduction

**Objective:** identify where fashion returns concentrate, translate the finding into a product intervention, and estimate the order-of-magnitude impact.

### Dataset
This case uses the **Data Mining Cup 2016 online-fashion returns dataset**, publicly mirrored on Kaggle as *Predicting Returns of Discounted Articles Sales*. It contains about **2.33M real order line items** from an anonymized fashion retailer, with order attributes, product attributes including size/color, and `returnQuantity`. The dataset is CC0 according to its Kaggle page. The raw data is not redistributed here; the notebook downloads it at runtime.

**Kaggle dataset:** https://www.kaggle.com/datasets/oscarm524/predicting-returns-of-discounted-articles-sales

**Reproducibility reference:** https://github.com/myBytesResearch/fashion-returns-analysis

> **Important:** Myntra's internal return data is not public. The analysis therefore uses a public fashion-retailer dataset as a directional proxy, not as a claim about Myntra's exact SKU/category return rates.

## 1. Executive takeaways

- The public dataset has an overall return rate of about **52.0%** at the order-line level.
- **Size bracketing** is a major, concrete lever: **16.6%** of line items are brackets, with a **73% return rate**, and they account for **23.5% of all returns**.
- Only the brackets where **exactly one size is ultimately kept** are clearly addressable by size recommendation. A leakage-free personalized recommender hit the kept size **28.8%** of the time versus **19.1%** for an item-level baseline.
- The measured recommender captured roughly **2% of all returns**, with a theoretical ceiling of **6.9%** from this order/return signal alone.
- **Product recommendation:** start with a size-and-fit layer rather than leading with AR: personalized size recommendation + corrected garment measurements + review-derived fit signals. Add AR selectively for visually/shape-sensitive categories.
- For a simple scale example of **1M order lines**, a 2% reduction in returns implies about **10,400 avoided returns** at the 52% baseline. Using a public cost estimate of **€3.60–€5.14 per return**, that is roughly **€37k–€53k** gross savings before intervention cost.

The 2% intervention effect is a measured result from the public DMC-based analysis; the 1M-order calculation is a back-of-envelope scaling exercise, not a Myntra forecast.

In [ ]:
# Optional: install the public-data loader in a fresh environment
# %pip install -q kagglehub pandas numpy matplotlib seaborn

import os, glob, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

In [ ]:
# Download the dataset from Kaggle.
# If Kaggle asks for credentials, follow Kaggle's authentication instructions.
import kagglehub

dataset_path = kagglehub.dataset_download(
    'oscarm524/predicting-returns-of-discounted-articles-sales'
)
print('Downloaded to:', dataset_path)
print('\nFiles:')
for p in Path(dataset_path).rglob('*'):
    if p.is_file():
        print(p)

## 2. Load and inspect the raw order data

The competition data uses semicolon-separated text files and `NA` for missing values. Because mirrors can expose slightly different filenames, the code below searches for the training file instead of hard-coding one path.

In [ ]:
files = list(Path(dataset_path).rglob('*'))
train_candidates = [p for p in files if p.name.lower() in {'orders_train.txt', 'orders_train.csv'}]
if not train_candidates:
    train_candidates = [p for p in files if 'train' in p.name.lower() and p.suffix.lower() in {'.txt', '.csv'}]
if not train_candidates:
    raise FileNotFoundError('Could not find the training order file in the Kaggle download.')

train_file = train_candidates[0]
print('Using:', train_file)

df = pd.read_csv(train_file, sep=';', na_values=['NA'], low_memory=False)
print('Shape:', df.shape)
display(df.head())
display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)}))

## 3. Create the return flag and discover relevant fields

The exact column names can vary by mirror. The helper below identifies the return, size, and product-group fields using name patterns and prints the matches before analysis.

In [ ]:
def find_cols(patterns):
    hits = []
    for c in df.columns:
        cl = c.lower()
        if any(re.search(p, cl) for p in patterns):
            hits.append(c)
    return hits

return_cols = find_cols([r'return.*quantity', r'quantity.*return', r'^return$'])
size_cols = find_cols([r'size', r'sizecode', r'clothing.*size'])
group_cols = find_cols([r'product.*group', r'productgroup', r'category', r'group'])

print('Return candidates:', return_cols)
print('Size candidates:', size_cols)
print('Product/category candidates:', group_cols)

return_col = return_cols[0]
df['is_returned'] = pd.to_numeric(df[return_col], errors='coerce').fillna(0).gt(0).astype(int)
print('Overall return rate:', df['is_returned'].mean())

## 4. Which product groups/categories drive returns?

This is the key Myntra-style merchandising cut: compare **volume** and **return rate**. A category with a high return rate but tiny volume is less important than a category that combines high return rate with lots of orders.

In [ ]:
group_col = group_cols[0]
category_summary = (
    df.groupby(group_col)
      .agg(order_lines=('is_returned', 'size'),
           returns=('is_returned', 'sum'),
           return_rate=('is_returned', 'mean'))
      .assign(return_share=lambda x: x['returns'] / x['returns'].sum())
      .sort_values('return_rate', ascending=False)
)
display(category_summary.head(15))

top = category_summary.head(10).sort_values('return_rate')
ax = top['return_rate'].mul(100).plot(kind='barh', figsize=(8,5))
ax.set_xlabel('Return rate (%)')
ax.set_ylabel(group_col)
ax.set_title('Highest-return product groups')
plt.tight_layout()
plt.show()

## 5. Which sizes drive returns?

The most actionable sizing signal is not simply 'XL returns more than M'. It is **bracketing**: the same product is ordered in multiple sizes. That behavior is much closer to a product problem than a customer-level preference and gives a direct target for a size recommender.

In [ ]:
size_col = size_cols[0]
size_summary = (
    df.dropna(subset=[size_col])
      .groupby(size_col)
      .agg(order_lines=('is_returned', 'size'),
           returns=('is_returned', 'sum'),
           return_rate=('is_returned', 'mean'))
      .sort_values('return_rate', ascending=False)
)
display(size_summary)

ax = size_summary['return_rate'].mul(100).plot(kind='bar', figsize=(8,5))
ax.set_ylabel('Return rate (%)')
ax.set_xlabel('Size')
ax.set_title('Return rate by size')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Publicly reproduced sizing benchmark

The companion analysis of the same DMC 2016 data reports: **16.6%** of line items are size brackets; brackets have a **73%** return rate and represent **23.5%** of all returns. Of those brackets, **40.5%** keep exactly one size, which is the clearest addressable sizing pool. A leakage-free personalized recommender hit the actually kept size **28.8%** of the time vs **19.1%** for an item-level baseline. The measured capture was about **2% of all returns**, with a **6.9% theoretical ceiling**.

Source: myBytes Research, *What size recommendation actually does against returns* (June 2026): https://mybytes.com/en/research/fashion-size-recommendation

In [ ]:
# Back-of-envelope impact for 1,000,000 order lines.
orders = 1_000_000
baseline_return_rate = 0.52
returns = orders * baseline_return_rate
capture_rate = 0.02
avoided_returns = returns * capture_rate
cost_low, cost_high = 3.60, 5.14

impact = pd.DataFrame({
    'metric': ['Order lines', 'Baseline returns', 'Avoided returns @ 2%',
               'Gross savings @ €3.60/return', 'Gross savings @ €5.14/return'],
    'value': [orders, returns, avoided_returns,
              avoided_returns * cost_low,
              avoided_returns * cost_high]
})
display(impact)

## 6. Product intervention recommendation

### MVP: 'Fit Confidence' on the product page

1. **Personalized size recommendation** — recommend a size from the customer's previous kept/returned sizes, with a fallback to item/category history for new users.
2. **Better size chart** — show garment measurements, not just S/M/L/XL labels; flag SKUs whose observed kept-size pattern is materially different from the chart.
3. **Review-based sizing** — summarize reviews into signals such as 'runs small', 'true to size', 'oversized', and fit by body/garment attribute where enough review volume exists.
4. **AR try-on as a secondary lever** — test it first in visually/shape-sensitive categories where 'how it looks on me' is a major source of uncertainty. AR should not be the first fix for a broken size table.

### Experiment design
- Randomize eligible traffic into control vs Fit Confidence treatment.
- Primary KPI: return rate, especially size/fit-related returns.
- Guardrails: conversion, cancellation rate, customer satisfaction, exchange rate, and AOV.
- Segment results by category, size, new vs repeat customer, and SKU.
- Do not infer causality from observational return data alone; the intervention effect should be measured with an A/B test.

## 7. Economics and decision rule

A public reproducibility study on the same dataset estimates fashion-return cost at **€3.60–€5.14 per return**, including handling plus expected value loss. With a hypothetical €1 intervention cost and a 30% prevention rate, the break-even return cost is about **€3.33**. This means return prevention is economically sensitive to how Myntra accounts for markdown/B-stock/destruction losses.

Source: myBytes Research, *What fashion returns really cost*: https://mybytes.com/research/retourenkosten-mode

### Recommendation
**Prioritize size/fit UX over a broad AR rollout.** The public data shows a measurable sizing/bracketing pool, while the strongest honest estimate for a purchase-history-only recommender is modest. Use the pilot to learn which categories have enough addressable fit uncertainty to justify deeper investment in AR or richer body/garment measurements.

## 8. Caveats

- The public dataset is from an anonymized online fashion retailer, not Myntra; category-level results should be treated as directional.
- The raw DMC data is not redistributed with this notebook because the companion research notes it is third-party data; the notebook downloads it from Kaggle when run.
- The 2% return-capture figure is a measured result of a purchase-history size recommender in the companion analysis; the proposed combined intervention (size chart + review signals + optional AR) has **not** been measured here.
- The 1M-order savings example is intentionally simple and should be replaced with Myntra's actual order volume, contribution margin, return-processing cost, and markdown/write-off rates.